# 📊 AI-Powered Sales Analytics RAG Assistant
Python · SQL · Pandas · LangChain · Gemini · FAISS

**Run the cells top to bottom.** Runtime → Run all works after you add your API key (Step 1).

## Step 1 – Get your free Gemini API key
1. Open https://aistudio.google.com/apikey and sign in with Google.
2. Click **Create API key** → choose/create a project → copy the key.
3. In Colab click the 🔑 **Secrets** icon (left sidebar) → **Add new secret** → Name: `GOOGLE_API_KEY`, Value: your key → turn on **Notebook access**.

Never paste your key into a notebook you will share or push to GitHub.

In [ ]:
!pip -q install streamlit pandas openpyxl pypdf faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-google-genai

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")
print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

## Step 2 – Write the RAG core module
This creates `rag_core.py` (loading, cleaning, chunking, embedding, FAISS, answering).

In [1]:
%%writefile rag_core.py
"""Core RAG logic for the AI-Powered Sales Analytics Assistant."""
import pandas as pd
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# If Google retires a model, pick a current one in https://aistudio.google.com
CHAT_MODEL = "gemini-3.6-flash"
EMBED_MODEL = "models/gemini-embedding-001"
DIMENSIONS = ["Month", "Product", "Category", "Region", "Customer"]


# ---------- 1. Load & clean ----------
def load_sales_file(name, fileobj):
    """Read a CSV or Excel file into a DataFrame."""
    if name.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(fileobj)
    return pd.read_csv(fileobj)


def prepare_sales_df(df):
    """Standardise columns: parse dates, add Month, make sure Revenue exists."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    date_col = next((c for c in df.columns if "date" in c.lower()), None)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df["Month"] = df[date_col].dt.to_period("M").astype(str)
    if "Revenue" not in df.columns:
        if {"Quantity", "Unit_Price"}.issubset(df.columns):
            df["Revenue"] = df["Quantity"] * df["Unit_Price"]
        else:
            raise ValueError("Need a 'Revenue' column, or 'Quantity' and 'Unit_Price'.")
    return df.dropna(subset=["Revenue"])


def load_business_doc(name, fileobj):
    """Read a PDF / TXT / MD business document into a Document."""
    if name.lower().endswith(".pdf"):
        text = "\n".join((p.extract_text() or "") for p in PdfReader(fileobj).pages)
    else:
        text = fileobj.read().decode("utf-8", errors="ignore")
    return [Document(page_content=text, metadata={"source": name})]


# ---------- 2. Turn tables into searchable text ----------
def sales_to_documents(df):
    docs = []
    total = df["Revenue"].sum()
    kpi = [f"OVERALL KPIs: total revenue = {total:,.2f}; rows/orders = {len(df)}; "
           f"average order value = {df['Revenue'].mean():,.2f}."]
    if "Month" in df:
        kpi.append(f"Data covers {df['Month'].min()} to {df['Month'].max()}.")
    kpi.append("Columns: " + ", ".join(df.columns))
    docs.append(Document(page_content=" ".join(kpi), metadata={"source": "sales_summary:kpis"}))

    for dim in DIMENSIONS:
        if dim not in df.columns:
            continue
        g = df.groupby(dim)["Revenue"].agg(["sum", "count"])
        if dim == "Month":
            g = g.sort_index()
            g["mom_%"] = (g["sum"].pct_change() * 100).round(1)
        else:
            g = g.sort_values("sum", ascending=False).head(40)
        lines = [f"REVENUE BY {dim.upper()}:"]
        for k, r in g.iterrows():
            extra = f", MoM growth={r['mom_%']}%" if "mom_%" in g.columns and pd.notna(r["mom_%"]) else ""
            lines.append(f"{k}: revenue={r['sum']:,.2f}, orders={int(r['count'])}{extra}")
        docs.append(Document(page_content="\n".join(lines), metadata={"source": f"sales_summary:{dim}"}))

    if {"Month", "Category"}.issubset(df.columns):
        pv = df.pivot_table(index="Month", columns="Category", values="Revenue", aggfunc="sum").round(0)
        docs.append(Document(page_content="MONTHLY REVENUE BY CATEGORY:\n" + pv.to_string(),
                             metadata={"source": "sales_summary:month_x_category"}))

    for i in range(0, min(len(df), 200), 20):  # a few raw rows for detail questions
        docs.append(Document(page_content="SAMPLE ORDER ROWS:\n" + df.iloc[i:i + 20].to_csv(index=False),
                             metadata={"source": f"sales_rows:{i}-{i + 19}"}))
    return docs


# ---------- 3. Index & answer ----------
def build_index(sales_df, business_docs, api_key):
    docs = sales_to_documents(sales_df) + list(business_docs)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
    chunks = splitter.split_documents(docs)
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, google_api_key=api_key)
    return FAISS.from_documents(chunks, embeddings), len(chunks)


PROMPT = """You are a senior sales analyst. Answer the question using ONLY the context below.
Quote exact numbers, name products/customers/regions, and describe trends clearly.
If the context does not contain the answer, say so instead of guessing.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""


def _as_text(content):
    if isinstance(content, list):
        return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return content


def ask(question, store, api_key, k=6):
    hits = store.similarity_search(question, k=k)
    context = "\n\n---\n\n".join(d.page_content for d in hits)
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, google_api_key=api_key, temperature=0.2)
    reply = llm.invoke(PROMPT.format(context=context, question=question))
    return _as_text(reply.content), hits

Overwriting rag_core.py


## Step 3 – Load data
Keep `USE_SAMPLE = True` to try a generated demo dataset, or set it to `False` to upload your own sales file (CSV/Excel) and optional business documents (PDF/TXT).

In [ ]:
%%writefile generate_sample_data.py
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
catalog = {  # product: (category, price)
    "Laptop": ("Electronics", 900), "Smartphone": ("Electronics", 650),
    "Headphones": ("Electronics", 120), "Office Chair": ("Furniture", 210),
    "Standing Desk": ("Furniture", 480), "Notebook Pack": ("Stationery", 15),
    "Printer": ("Electronics", 260), "Monitor": ("Electronics", 300),
}
customers = [f"Customer_{i:02d}" for i in range(1, 31)]
regions = ["North", "South", "East", "West"]
n = 1500
products = rng.choice(list(catalog), n)
dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
df = pd.DataFrame({
    "Order_ID": [f"ORD{1000 + i}" for i in range(n)],
    "Date": dates,
    "Customer": rng.choice(customers, n),
    "Product": products,
    "Category": [catalog[p][0] for p in products],
    "Region": rng.choice(regions, n, p=[0.3, 0.2, 0.25, 0.25]),
    "Quantity": rng.integers(1, 8, n),
})
df["Unit_Price"] = [round(catalog[p][1] * rng.uniform(0.9, 1.1), 2) for p in products]
df["Revenue"] = (df["Quantity"] * df["Unit_Price"]).round(2)
df.sort_values("Date").to_csv("sales_data.csv", index=False)

open("business_notes.txt", "w").write("""ACME Retail - Business Notes 2025
Goal: grow annual revenue by 15% versus 2024 and keep Electronics above 60% of sales.
Strategy: push Standing Desk and Monitor bundles in Q3; run a back-to-school Notebook Pack promotion in July.
The West region is the priority for expansion; the South region has weaker marketing coverage.
Top customers receive a 5% loyalty discount when they exceed 20 orders per year.
""")
print("Created sales_data.csv and business_notes.txt")

In [ ]:
USE_SAMPLE = True

import io, pandas as pd
from rag_core import load_sales_file, prepare_sales_df, load_business_doc

business_docs = []
if USE_SAMPLE:
    !python generate_sample_data.py
    raw = pd.read_csv("sales_data.csv")
    business_docs = load_business_doc("business_notes.txt", open("business_notes.txt", "rb"))
else:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith((".csv", ".xlsx", ".xls")):
            raw = load_sales_file(name, io.BytesIO(data))
        else:
            business_docs += load_business_doc(name, io.BytesIO(data))

df = prepare_sales_df(raw)
df.head()

## Step 4 – Explore with Pandas and SQL

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
df.to_sql("sales", conn, index=False, if_exists="replace")

print("Top 5 products by revenue")
display(pd.read_sql("SELECT Product, ROUND(SUM(Revenue),2) AS revenue, COUNT(*) AS orders FROM sales GROUP BY Product ORDER BY revenue DESC LIMIT 5", conn))
print("Revenue by region")
display(pd.read_sql("SELECT Region, ROUND(SUM(Revenue),2) AS revenue FROM sales GROUP BY Region ORDER BY revenue DESC", conn))
df.groupby("Month")["Revenue"].sum().plot(kind="bar", figsize=(10, 3), title="Monthly revenue");

## Step 5 – Build the knowledge base (chunks → Gemini embeddings → FAISS)

In [ ]:
import os
from rag_core import build_index, ask
API_KEY = os.environ["GOOGLE_API_KEY"]
store, n_chunks = build_index(df, business_docs, API_KEY)
print("Indexed chunks:", n_chunks)

## Step 6 – Ask questions

In [ ]:
questions = [
    "What is the total revenue and average order value?",
    "Which product generates the most revenue?",
    "Which region is performing the weakest?",
    "Who are the top 5 customers by revenue?",
    "How is monthly revenue trending? Mention the best and worst months.",
    "What does the business strategy say about the West region?",
]
for q in questions:
    answer, sources = ask(q, store, API_KEY)
    print("Q:", q, "\nA:", answer, "\n" + "-" * 80)

In [ ]:
# Your own question:
answer, sources = ask("Which category should we invest in next quarter and why?", store, API_KEY)
print(answer)
print("\nSources:", [s.metadata["source"] for s in sources])

## Step 7 – Export cleaned data for Power BI
Download these files, then in Power BI Desktop use **Get data → Text/CSV**.

In [ ]:
df.to_csv("powerbi_sales_clean.csv", index=False)
df.groupby(["Month", "Category", "Product", "Region"], as_index=False)["Revenue"].sum().to_csv("powerbi_sales_summary.csv", index=False)
from google.colab import files
files.download("powerbi_sales_clean.csv"); files.download("powerbi_sales_summary.csv")

## Step 8 – (Optional) Launch the Streamlit web app inside Colab
Run the next two cells. The second prints a URL – open it and, when asked for a password/endpoint IP, paste the IP printed above it.

In [ ]:
%%writefile app.py
"""Streamlit UI: upload sales data + business docs, then chat with them."""
import os
import streamlit as st
from rag_core import (load_sales_file, prepare_sales_df, load_business_doc,
                      build_index, ask)

st.set_page_config(page_title="Sales Analytics RAG Assistant", page_icon="📊", layout="wide")
st.title("📊 AI-Powered Sales Analytics RAG Assistant")
st.caption("Upload sales data and business documents, then ask about revenue, products, customers and trends.")

with st.sidebar:
    st.header("1. Setup")
    api_key = st.text_input("Gemini API key", type="password",
                            value=os.environ.get("GOOGLE_API_KEY", ""),
                            help="Free key: https://aistudio.google.com/apikey")
    sales_file = st.file_uploader("Sales data (CSV / Excel)", type=["csv", "xlsx", "xls"])
    doc_files = st.file_uploader("Business documents (PDF / TXT / MD)",
                                 type=["pdf", "txt", "md"], accept_multiple_files=True)
    top_k = st.slider("Chunks retrieved (k)", 3, 12, 6)
    build = st.button("Build knowledge base", type="primary")

if build:
    if not api_key or not sales_file:
        st.sidebar.error("Provide an API key and a sales file.")
    else:
        try:
            with st.spinner("Cleaning data, embedding and indexing..."):
                df = prepare_sales_df(load_sales_file(sales_file.name, sales_file))
                extra = [d for f in doc_files for d in load_business_doc(f.name, f)]
                store, n = build_index(df, extra, api_key)
            st.session_state.update(df=df, store=store, messages=[], api_key=api_key)
            st.sidebar.success(f"Indexed {n} chunks from {len(df):,} rows.")
        except Exception as e:
            st.sidebar.error(f"Failed: {e}")

if "store" not in st.session_state:
    st.info("👈 Add your API key, upload files and click **Build knowledge base**.")
    st.stop()

df = st.session_state["df"]
c1, c2, c3 = st.columns(3)
c1.metric("Total revenue", f"{df['Revenue'].sum():,.0f}")
c2.metric("Orders", f"{len(df):,}")
c3.metric("Avg order value", f"{df['Revenue'].mean():,.0f}")
if "Month" in df:
    st.bar_chart(df.groupby("Month")["Revenue"].sum(), height=220)

st.subheader("💬 Ask your data")
for m in st.session_state["messages"]:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

if q := st.chat_input("e.g. Which product earned the most revenue last quarter?"):
    st.session_state["messages"].append({"role": "user", "content": q})
    with st.chat_message("user"):
        st.markdown(q)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                answer, hits = ask(q, st.session_state["store"], st.session_state["api_key"], top_k)
            except Exception as e:
                answer, hits = f"Error: {e}", []
        st.markdown(answer)
        if hits:
            with st.expander("Sources used"):
                for h in hits:
                    st.markdown(f"**{h.metadata.get('source')}**")
                    st.code(h.page_content[:600])
    st.session_state["messages"].append({"role": "assistant", "content": answer})

In [ ]:
!npm install -s localtunnel
!streamlit run app.py --server.port 8501 &>/content/streamlit_logs.txt &
import time; time.sleep(6)
print("Password / endpoint IP for the tunnel page:")
!curl -s https://ipv4.icanhazip.com
!npx localtunnel --port 8501